# SuperTrend

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [SuperTrend](#supertrend)
  - [Backtesting](#supertrend-backtesting)
  - [Grid search](#supertrend-grid-search)
  - [Walk-forward analysis](#supertrend-walk-forward)
  - [Monte Carlo simulations](#supertrend-monte-carlo)
  - [Live signals](#supertrend-live-signals)
- [Inverse SuperTrend](#inv-supertrend)
  - [Backtesting](#inv-supertrend-backtesting)
  - [Grid search](#inv-supertrend-grid-search)
  - [Walk-forward analysis](#inv-supertrend-walk-forward)
  - [Monte Carlo simulations](#inv-supertrend-monte-carlo)
  - [Live signals](#inv-supertrend-live-signals)
- [Adaptive SuperTrend](#adp-supertrend)
  - [Backtesting](#adp-supertrend-backtesting)
  - [Grid search](#adp-supertrend-grid-search)
  - [Walk-forward analysis](#adp-supertrend-walk-forward)
  - [Monte Carlo simulations](#adp-supertrend-monte-carlo)
  - [Live signals](#adp-supertrend-live-signals)
- [Live-mode for several strategies simultaneously](#live-mode-several)
  - [From CLI](#live-mode-several-cli)
  - [From a notebook cell](#live-mode-several-notebook)

SuperTrend Trend-Following \
Pure trend strategy with built-in ATR trailing stop. \
It uses ATR(14) for dynamic stops/tolerance (adapts to volatility). \
Extremely popular in crypto perpetuals for all timeframes; maximizes profit by riding trends while cutting losses fast.

How SuperTrend Algorithm Determines Entry/Exit:
- Combines ATR volatility bands with trend direction.
- Long Entry: SuperTrend flips from red to green (price closes above upper band).
- Short Entry: SuperTrend flips from green to red (price closes below lower band).
- Exit: SuperTrend flips opposite OR the built-in ATR trailing stop is hit (the SuperTrend line itself acts as dynamic stop).
- Extremely clean, low-lag, and maximizes trend capture while protecting capital.

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search, oracle_ceiling
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("supertrend")   # engine/strategy_configurator.py (SupertrendParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic SupertrendParams().
# A foreign key raises TypeError here, not a silent no-op.
STRATEGY_OVERRIDES = {}      # e.g. {"supertrend_period": 7, "supertrend_mult": 2.5}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

# Report exactly what data + which configs are in force downstream (manual or automatic).
_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

<a id="supertrend"></a>
## SuperTrend

<a id="supertrend-backtesting"></a>
### Backtesting

In [ ]:
# Import SuperTrend strategy
from engine.strategies import SuperTrendStrategy
STRATEGY = SuperTrendStrategy

In [ ]:
# Backtest SuperTrend strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="supertrend")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# SuperTrend strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="supertrend-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per supertrend_period × supertrend_mult cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"supertrend_mult", "supertrend_period"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="supertrend_period", columns="supertrend_mult", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="supertrend_mult", y="supertrend_period", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs supertrend_period × supertrend_mult swept in the grid above — nothing to plot.")

<a id="supertrend-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the per-fold winning params.
# Each fold's chosen indicators are recomputed and shown only over that fold's test window;
# the step at fold boundaries is the re-optimisation. See wf.folds_frame() for the params.

prepared_wf = df.copy()
prepared_wf['supertrend'] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ['supertrend']] = prep.loc[seg, ['supertrend']]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold params").show()

<a id="supertrend-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="supertrend-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy supertrend --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="inv-supertrend"></a>
## Inverse SuperTrend

<a id="inv-supertrend-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse SuperTrend strategy
from engine.strategies import InverseSuperTrendStrategy
STRATEGY = InverseSuperTrendStrategy

In [ ]:
# Backtest Inverse SuperTrend strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="supertrend_inv")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# Inverse SuperTrend strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-supertrend-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per supertrend_period × supertrend_mult cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"supertrend_mult", "supertrend_period"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="supertrend_period", columns="supertrend_mult", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="supertrend_mult", y="supertrend_period", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs supertrend_period × supertrend_mult swept in the grid above — nothing to plot.")

<a id="inv-supertrend-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the per-fold winning params.
# Each fold's chosen indicators are recomputed and shown only over that fold's test window;
# the step at fold boundaries is the re-optimisation. See wf.folds_frame() for the params.

prepared_wf = df.copy()
prepared_wf['supertrend'] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ['supertrend']] = prep.loc[seg, ['supertrend']]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold params").show()

<a id="inv-supertrend-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="inv-supertrend-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy supertrend_inv --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="adp-supertrend"></a>
## Adaptive SuperTrend

On each bar, ADX is checked. \
If ADX ≥ 25 (trending), a SuperTrend flip triggers a trade in the same direction (trend-following, normal SuperTrend). \
If ADX < 25 (ranging), the same flip triggers the opposite direction (mean-reversion, inverse SuperTrend). \
Exits also respect the current regime — so if the market shifts from trending to ranging mid-trade, the exit condition matches what made sense at entry.

The adx_threshold parameter is now a SupertrendParams knob — tune it from the manual chapter (STRATEGY_OVERRIDES = {"adx_threshold": 20}). \
It controls the regime switch:
- lower (20) makes it more aggressive about calling "trending"
- higher (30) makes it pickier

<a id="adp-supertrend-backtesting"></a>
### Backtesting

In [ ]:
# Import Adaptive SuperTrend strategy
from engine.strategies import AdaptiveSuperTrendStrategy
STRATEGY = AdaptiveSuperTrendStrategy

In [ ]:
# Backtest Adaptive SuperTrend strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="supertrend_adaptive")

In [ ]:
# Theoretical ceiling
# For OHLCV-only crypto: ~5–15% is a realistic edge; under ~5% is noise.
_ceil_bps, _chain = oracle_ceiling(df, cost_bps=TRADING_CONFIG.total_cost_bps())
_skill = result.total_pnl_bps / _ceil_bps * 100 if _ceil_bps else 0.0
print(f"Theoretical ceiling : {_ceil_bps:+,.0f} bps  (full look-ahead, {TRADING_CONFIG.total_cost_bps():.0f} bps cost, {max(0, len(_chain) - 1)} trades)")
print(f"Strategy P&L   : {result.total_pnl_bps:+,.0f} bps")
print(f"Skill ratio    : {_skill:.1f}%  captured of what was theoretically possible")

In [ ]:
# Adaptive SuperTrend strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="adp-supertrend-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per supertrend_period × supertrend_mult cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"supertrend_mult", "supertrend_period"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="supertrend_period", columns="supertrend_mult", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="supertrend_mult", y="supertrend_period", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs supertrend_period × supertrend_mult swept in the grid above — nothing to plot.")

<a id="adp-supertrend-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"supertrend_period": [7, 10, 14], "supertrend_mult": [2.0, 3.0, 4.0]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the per-fold winning params.
# Each fold's chosen indicators are recomputed and shown only over that fold's test window;
# the step at fold boundaries is the re-optimisation. See wf.folds_frame() for the params.

prepared_wf = df.copy()
prepared_wf['supertrend'] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ['supertrend']] = prep.loc[seg, ['supertrend']]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold params").show()

<a id="adp-supertrend-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="adp-supertrend-live-signals"></a>
### Live signals

Live mode:
- runs the same strategy / config / costs as the backtest above
- generates signals: tells you when to enter / exit
- does not place orders
- the chart auto-refreshes every poll_seconds

Signal notification + sound:
- alerts you on each new entry/exit
- the first poll primes silently; alerts start from the next new signal
- browser = a banner + beep right in this cell's output (Safari/Chrome) \
  Runs via the notebook cell, not via CLI.
- desktop = a native macOS notification
- telegram reaches your phone by setting TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID in the environment

From the CLI:
- a terminal running the same loop
- prints a file link to the auto-refreshing chart
- add --notify to alert on each new signal: \
  python -m engine --strategy supertrend_adaptive --mode live --interval 15 --poll 30 --notify desktop,telegram

From a notebook cell:
- prints a clickable chart link
- run() blocks the execution of the rest of the notebook until stopped

In [ ]:
live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
    notifiers="browser,desktop",        # ← edit here: browser / desktop / telegram, or None to disable
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="live-mode-several"></a>
## Live-mode for several strategies simultaneously

<a id="live-mode-several-cli"></a>
### From CLI

- one terminal per strategy
- no need to set --db/--save: the defaults already write per-strategy paths under data/live/ (data/live/<strategy>.db and data/live/<symbol><interval><strategy>.html), so concurrent runs don't clobber each other

In [ ]:
python -m engine --strategy supertrend          --mode live --interval 15 &
python -m engine --strategy supertrend_inv      --mode live --interval 15 &
python -m engine --strategy supertrend_adaptive --mode live --interval 15 &

<a id="live-mode-several-notebook"></a>
### From a notebook cell

In [ ]:
# Multi-strategy live: runs all three Supertrend variants at once, each wired to
# the notebook's configured symbol / interval / params / costs / exit (the three
# configurators), with a separate chart + DB per strategy.
import threading
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategies import SuperTrendStrategy
from engine.strategies.supertrend_inv import InverseSuperTrendStrategy
from engine.strategies.supertrend_adaptive import AdaptiveSuperTrendStrategy

strategies = [
    SuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY),
    InverseSuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY),
    AdaptiveSuperTrendStrategy(STRATEGY_CONFIG, exit_policy=EXIT_POLICY),
]

threads = []
for strat in strategies:
    engine = LiveEngine(
        strategy=strat,
        symbol=SYMBOL,
        interval=INTERVAL,
        num_candles=500,
        poll_seconds=30,
        trading_config=TRADING_CONFIG,
        chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strat.name}.html"),
        db_path=str(LIVE_DIR / f"{strat.name}.db"),   # separate DB per strategy
        notifiers="browser,desktop",   # ← browser / desktop / telegram, or None
    )
    t = threading.Thread(target=engine.run, name=strat.name, daemon=True)
    threads.append(t)
    t.start()
    print(f"Started: {strat.name}")

# Blocks until you interrupt the kernel
for t in threads:
    t.join()